In [1]:
import numpy as np
import pandas as pd
import psycopg2
import pytz
from tqdm import tqdm
from datetime import timedelta
from dateutil.relativedelta import relativedelta

In [2]:
sales = pd.read_csv('pedidos-1743765883268.csv', sep=';', encoding='latin1')

In [3]:
df = sales[['account_id', 'sales_channel_id']].value_counts()
df = pd.DataFrame(df).reset_index()

In [4]:
# EPOCHS = 1000
# BATCH_SIZE = 32
# VALID_SPLIT = 0.1

In [5]:
SAO_PAULO_TZ = pytz.timezone('America/Sao_Paulo')
LOOKBACK = 1
# SEASONAL_PERIODS = (24, 24*7)
# COVERAGE = 0.33
END_DATE = pd.to_datetime(pd.to_datetime(sales['created_date'].max()).strftime("%Y-%m-%d %H:00:00"))
# START_DATE = END_DATE - timedelta(hours=END_DATE.hour)
START_DATE = END_DATE - timedelta(hours=23)
# TEST_SIZE = int((END_DATE - START_DATE).seconds / 60**2 + 1)

In [6]:
# OOT_DATE = START_DATE + timedelta(hours=23)

In [7]:
# OOT_PERIODS = int((OOT_DATE - END_DATE).seconds / 60**2 + 1)

In [8]:
# oot_dates = pd.DatetimeIndex([END_DATE+timedelta(hours=h) for h in range(1, OOT_PERIODS)], freq='h')
# df_oot = pd.DataFrame(index=oot_dates)

In [9]:
# df.drop('count', axis=1, inplace=True)
# for account_id in df['account_id'].unique():
#     df = pd.concat([df, pd.DataFrame({'account_id': [account_id], 'sales_channel_id': ['ALL']})], ignore_index=True)

In [10]:
# df

In [11]:
id_pairs = list(zip(df['account_id'], df['sales_channel_id']))
# id_pairs = list(zip(df_filtered['account_id'], df_filtered['sales_channel_id']))

In [12]:
weights_chan = {}
for account_id, sales_channel_id in tqdm(id_pairs):
    # if sales_channel_id == 'ALL':
    #     cond = (sales['account_id'] == account_id) & \
    #            (sales['status'].notna())
    # else:
    #     cond = (sales['account_id'] == account_id) & \
    #            (sales['sales_channel_id'] == sales_channel_id) & \
    #            (sales['status'].notna())
    cond = (sales['account_id'] == account_id) & \
           (sales['sales_channel_id'] == sales_channel_id) & \
           (sales['status'].notna())

    df_client = sales[cond].drop(['account_id', 'sales_channel_id'], axis=1)
    df_client['created_date'] = pd.to_datetime(df_client['created_date'], format='%Y-%m-%d %H:%M:%S.%f %z')
    df_client = df_client.sort_values('created_date').reset_index(drop=True)
    df_client['created_date'] = df_client['created_date'].dt.strftime("%Y-%m-%d %H:00:00").reset_index(drop=True)

    df_client_mod = df_client.groupby('created_date').agg(price_total_agg=('price_total', 'sum'), n_orders=('created_date', 'count'))
    df_client_mod.index = pd.to_datetime(df_client_mod.index, format='%Y-%m-%d %H:00:00')

    end_date = pd.to_datetime(END_DATE, format='%Y-%m-%d %H:%M:%S')
    start_date = end_date - relativedelta(months=LOOKBACK)
    date_index = pd.Series(pd.date_range(start=start_date, end=end_date, freq='h', name='created_date'))
    df_client_mod = pd.merge(date_index, df_client_mod, how='left', on='created_date').set_index('created_date')
    
    weights_chan[account_id] = {} if account_id not in weights_chan else weights_chan[account_id]
    
    df = df_client_mod['n_orders'].fillna(0)
    df = df.loc[df.index < START_DATE].copy()
    
    weights_chan[account_id][sales_channel_id] = df.median()

100%|██████████| 277/277 [05:25<00:00,  1.17s/it]


In [13]:
weights_df = pd.DataFrame(weights_chan)

weights_df

,5f5ed10c-7792-4080-970c-b11398e3f58f,9d39683a-47b3-4a18-942d-f1d741a47e8c,05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6,74abffac-6542-4eff-8eae-cc85463a5d02,f9129295-bb08-4330-b60c-9f0beadda521,60c8f748-e29d-4fce-9e66-584117ca2baf,40074fa8-e02b-4c91-a819-fe1e639b540d,173e2b27-8bc6-44d1-b581-a1913d6b1894,478cd985-f4fc-45b4-ac71-bf78cf11e07b,bfcbb283-4e53-41fd-9b31-aa818f7f23ce,...,b009c05b-326e-4ebe-988c-8d2229ec3a4c,22fde42e-45d7-46cd-9d6f-3a4dbabbc579,4f1c8b04-eb4b-4887-8c69-a096f0fda3ab,3dc15e4b-b60d-4c49-aae8-97cf9664af51,423a069b-27b8-44ed-9ef3-ca3cf9470970,0ad2f65b-ee0e-4d50-93e2-84b76282325f,aabe992c-cfad-4049-b5df-36c41f900a67,9d1296ca-9e4f-4840-b7b5-e12172acab78,457bce7b-9300-4c10-9a97-070b3c0d081d,2058f680-aace-4b31-aed4-f34397b75096
1,0.0,93.0,0.0,21.0,65.0,18.0,31.0,12.0,1.0,0.0,...,1.0,0.0,9.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0
9,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,...,NaN,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN
7,0.0,NaN,NaN,NaN,0.0,NaN,NaN,0.0,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN
3,0.0,NaN,62.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0.0,0.0,NaN
20,0.0,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,0.0,NaN,NaN,NaN,0.0,NaN,NaN,0.0,NaN,0.0,...,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
35,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,0.0,NaN,40.0,NaN,0.0,NaN,NaN,0.0,0.0,0.0,...,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN
5,0.0,NaN,6.0,NaN,0.0,NaN,NaN,1.0,0.0,0.0,...,NaN,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN
4,0.0,NaN,0.0,NaN,0.0,0.0,NaN,0.0,0.0,0.0,...,NaN,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN


In [14]:
weights_df = weights_df.stack().reset_index().rename(columns={'level_0': 'sales_channel_id', 'level_1': 'account_id', 0: 'weights'}).sort_values(by=['account_id', 'sales_channel_id'])

weights_df

,sales_channel_id,account_id,weights
2,1,05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6,0.0
65,3,05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6,62.0
135,4,05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6,0.0
117,5,05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6,6.0
101,6,05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6,40.0
...,...,...,...
253,36,f9129295-bb08-4330-b60c-9f0beadda521,0.0
274,38,f9129295-bb08-4330-b60c-9f0beadda521,6.0
217,39,f9129295-bb08-4330-b60c-9f0beadda521,0.0
272,42,f9129295-bb08-4330-b60c-9f0beadda521,0.0


In [15]:
weights_chan_df = weights_df.groupby('account_id')['weights'].apply(lambda x: x / x.sum()).fillna(0).reset_index().rename(columns={0: 'weights'}).drop('level_1', axis=1)

weights_chan_df = pd.concat([weights_df['sales_channel_id'].reset_index(drop=True), weights_chan_df], axis=1)

weights_chan_df

,sales_channel_id,account_id,weights
0,1,05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6,0.000000
1,3,05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6,0.574074
2,4,05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6,0.000000
3,5,05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6,0.055556
4,6,05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6,0.370370
...,...,...,...
272,36,f9129295-bb08-4330-b60c-9f0beadda521,0.000000
273,38,f9129295-bb08-4330-b60c-9f0beadda521,0.049180
274,39,f9129295-bb08-4330-b60c-9f0beadda521,0.000000
275,42,f9129295-bb08-4330-b60c-9f0beadda521,0.000000


In [16]:
weights_acc_df = weights_df.groupby('account_id')['weights'].sum() / weights_df.groupby('account_id')['weights'].sum().sum()

weights_acc_df

account_id
05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6    0.233261
0661e926-f219-456e-bde5-612fee6f891c    0.004320
0ad2f65b-ee0e-4d50-93e2-84b76282325f    0.006479
173e2b27-8bc6-44d1-b581-a1913d6b1894    0.028078
2058f680-aace-4b31-aed4-f34397b75096    0.000000
22fde42e-45d7-46cd-9d6f-3a4dbabbc579    0.000000
23797b7c-9df5-45e7-89cd-548bbb7bd98f    0.000000
3dc15e4b-b60d-4c49-aae8-97cf9664af51    0.000000
40074fa8-e02b-4c91-a819-fe1e639b540d    0.066955
423a069b-27b8-44ed-9ef3-ca3cf9470970    0.000000
457bce7b-9300-4c10-9a97-070b3c0d081d    0.000000
478cd985-f4fc-45b4-ac71-bf78cf11e07b    0.002160
4f1c8b04-eb4b-4887-8c69-a096f0fda3ab    0.025918
50409b57-b000-43d5-9358-44c3081c4132    0.002160
5dd8b3ee-6f1c-4ac9-bbc5-91829dde38eb    0.017279
5f5ed10c-7792-4080-970c-b11398e3f58f    0.000000
60c8f748-e29d-4fce-9e66-584117ca2baf    0.038877
60ca8452-5b14-481f-a637-8756f527aee8    0.000000
6276b87c-ebb2-11ed-a05b-0242ac120003    0.008639
6cd2fa8a-5bac-4cc0-b4df-bd2c7d29e71b    0.000000
7412074f-

In [17]:
weights_acc_df.sort_values(ascending=False)

account_id
f9129295-bb08-4330-b60c-9f0beadda521    0.263499
05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6    0.233261
9d39683a-47b3-4a18-942d-f1d741a47e8c    0.200864
40074fa8-e02b-4c91-a819-fe1e639b540d    0.066955
74abffac-6542-4eff-8eae-cc85463a5d02    0.045356
60c8f748-e29d-4fce-9e66-584117ca2baf    0.038877
173e2b27-8bc6-44d1-b581-a1913d6b1894    0.028078
4f1c8b04-eb4b-4887-8c69-a096f0fda3ab    0.025918
5dd8b3ee-6f1c-4ac9-bbc5-91829dde38eb    0.017279
e2de40dd-743b-4105-9077-3f343fea980d    0.015119
ad942d1a-15ab-4a0d-83a8-ae82183ece53    0.012959
6276b87c-ebb2-11ed-a05b-0242ac120003    0.008639
adc6fb62-09e7-44fd-a619-f787ecfd140b    0.006479
0ad2f65b-ee0e-4d50-93e2-84b76282325f    0.006479
0661e926-f219-456e-bde5-612fee6f891c    0.004320
ca35e88d-d191-4722-b90c-92f4a249869b    0.004320
775dcad4-0733-4c22-8dbd-cd3ce344e891    0.004320
92f116fe-5818-41db-9eb5-87445ad3818e    0.004320
50409b57-b000-43d5-9358-44c3081c4132    0.002160
478cd985-f4fc-45b4-ac71-bf78cf11e07b    0.002160
789ff5c4-

In [18]:
dbname = 'railway'
username = 'sinatra'
pwd = '781B3XjpeuqE'
hostname = 'monorail.proxy.rlwy.net'
port = 25096

connection = psycopg2.connect(database=dbname, user=username, password=pwd, host=hostname, port=port)
cursor = connection.cursor()

In [19]:
id_pairs_tbats = cursor.execute("select distinct account_id, channel from public.forecast where model='TBATS_10';")
res_tbats = cursor.fetchall()

id_pairs_gb = cursor.execute("select distinct account_id, channel from public.forecast where model='GradientBoosting_10';")
res_gb = cursor.fetchall()

id_pairs_lstm = cursor.execute("select distinct account_id, channel from public.forecast where model='LSTM_10';")
res_lstm = cursor.fetchall()

id_pairs_ens = cursor.execute("select distinct account_id, channel from public.forecast where model='Ensemble_10';")
res_ens = cursor.fetchall()

id_pairs_chronos = cursor.execute("select distinct account_id, channel from public.forecast where model='amazon/chronos-bolt-base:v2025-03-09:cl(370):q(0.3,0.7)' and channel!='ALL';")
res_chronos = cursor.fetchall()

id_pairs_median = cursor.execute("select distinct account_id, channel from public.forecast where model='MEDIAN_MAD_60' and channel!='ALL';")
res_median = cursor.fetchall()

In [20]:
id_pairs = (
    set(res_tbats)
    .intersection(set(res_gb))
    .intersection(set(res_lstm))
    .intersection(set(res_ens))
    .intersection(set(res_chronos))
    .intersection(set(res_median))
)

id_pairs = set(res_tbats).intersection(set(res_gb))

In [21]:
id_pairs

{('05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6', '1'),
 ('05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6', '3'),
 ('05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6', '4'),
 ('05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6', '5'),
 ('05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6', '6'),
 ('9d39683a-47b3-4a18-942d-f1d741a47e8c', '1'),
 ('9d39683a-47b3-4a18-942d-f1d741a47e8c', '2'),
 ('f9129295-bb08-4330-b60c-9f0beadda521', '1'),
 ('f9129295-bb08-4330-b60c-9f0beadda521', '10'),
 ('f9129295-bb08-4330-b60c-9f0beadda521', '11'),
 ('f9129295-bb08-4330-b60c-9f0beadda521', '13'),
 ('f9129295-bb08-4330-b60c-9f0beadda521', '15'),
 ('f9129295-bb08-4330-b60c-9f0beadda521', '16'),
 ('f9129295-bb08-4330-b60c-9f0beadda521', '17'),
 ('f9129295-bb08-4330-b60c-9f0beadda521', '18'),
 ('f9129295-bb08-4330-b60c-9f0beadda521', '19'),
 ('f9129295-bb08-4330-b60c-9f0beadda521', '2'),
 ('f9129295-bb08-4330-b60c-9f0beadda521', '20'),
 ('f9129295-bb08-4330-b60c-9f0beadda521', '21'),
 ('f9129295-bb08-4330-b60c-9f0beadda521', '22'),
 ('f9129295-bb08-4330-b60c-9f

In [22]:
gb_models = [f'GradientBoosting_{int(10*n)}' for n in range(1, 10)]
lstm_models = [f'LSTM_{int(10*n)}' for n in range(1, 10)]
tbats_models = [f'TBATS_{int(10*n)}' for n in range(1, 10)]
ens_models = [f'Ensemble_{int(10*n)}' for n in range(1, 10)]

In [23]:
id_pairs

{('05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6', '1'),
 ('05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6', '3'),
 ('05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6', '4'),
 ('05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6', '5'),
 ('05d9cc3a-decc-4fa2-b4e8-b44eef40e3b6', '6'),
 ('9d39683a-47b3-4a18-942d-f1d741a47e8c', '1'),
 ('9d39683a-47b3-4a18-942d-f1d741a47e8c', '2'),
 ('f9129295-bb08-4330-b60c-9f0beadda521', '1'),
 ('f9129295-bb08-4330-b60c-9f0beadda521', '10'),
 ('f9129295-bb08-4330-b60c-9f0beadda521', '11'),
 ('f9129295-bb08-4330-b60c-9f0beadda521', '13'),
 ('f9129295-bb08-4330-b60c-9f0beadda521', '15'),
 ('f9129295-bb08-4330-b60c-9f0beadda521', '16'),
 ('f9129295-bb08-4330-b60c-9f0beadda521', '17'),
 ('f9129295-bb08-4330-b60c-9f0beadda521', '18'),
 ('f9129295-bb08-4330-b60c-9f0beadda521', '19'),
 ('f9129295-bb08-4330-b60c-9f0beadda521', '2'),
 ('f9129295-bb08-4330-b60c-9f0beadda521', '20'),
 ('f9129295-bb08-4330-b60c-9f0beadda521', '21'),
 ('f9129295-bb08-4330-b60c-9f0beadda521', '22'),
 ('f9129295-bb08-4330-b60c-9f

In [24]:
df_forecast_metrics_norm = {}
scores = {}
# for model in gb_models + lstm_models + tbats_models + ens_models + ['amazon/chronos-bolt-base:v2025-03-09:cl(370):q(0.3,0.7)', 'MEDIAN_MAD_60']:
for model in gb_models + tbats_models:
    print('\n\n')
    print(model)
    df_forecast_metrics_norm[model] = {}
    
    for account_id, sales_channel_id in id_pairs:
        
        if sales_channel_id == 'ALL':
            cond = (sales['account_id'] == account_id) & \
                   (sales['status'].notna())
        else:
            cond = (sales['account_id'] == account_id) & \
                   (sales['sales_channel_id'] == int(sales_channel_id)) & \
                   (sales['status'].notna())

        df_client = sales[cond].drop(['account_id', 'sales_channel_id'], axis=1)
        df_client['created_date'] = pd.to_datetime(df_client['created_date'], format='%Y-%m-%d %H:%M:%S.%f %z')
        df_client = df_client.sort_values('created_date').reset_index(drop=True)
        df_client['created_date'] = df_client['created_date'].dt.strftime("%Y-%m-%d %H:00:00").reset_index(drop=True)

        df_client_mod = df_client.groupby('created_date').agg(price_total_agg=('price_total', 'sum'), n_orders=('created_date', 'count'))
        df_client_mod.index = pd.to_datetime(df_client_mod.index, format='%Y-%m-%d %H:00:00')

        end_date = pd.to_datetime(END_DATE, format='%Y-%m-%d %H:%M:%S')
        start_date = end_date - relativedelta(months=LOOKBACK)
        date_index = pd.Series(pd.date_range(start=start_date, end=end_date, freq='h', name='created_date'))
        df_client_mod = pd.merge(date_index, df_client_mod, how='left', on='created_date').set_index('created_date')\
        
        df_forecast_metrics_norm[model][account_id] = {} if account_id not in df_forecast_metrics_norm[model] else df_forecast_metrics_norm[model][account_id]
        for dataset in ['sales', 'orders']:
            if dataset == 'sales':
                col = 'price_total_agg'
            else:
                col = 'n_orders'
            
            df = df_client_mod[col].fillna(0)
            
            y_test = df.loc[(df.index >= START_DATE) & (df.index <= END_DATE)].copy()
        
            cursor.execute(f"select start, {dataset}_high, {dataset}_low, {dataset}_mean from public.forecast where account_id = '{account_id}' and channel = '{sales_channel_id}' and model = '{model}'")
            res = cursor.fetchall()
            
            df_forecast = pd.DataFrame(res, columns=['start', f'{dataset}_high', f'{dataset}_low', f'{dataset}_mean']) #[:len(y_test)].set_index(y_test.index)
            
            df_forecast[f'{dataset}_mean'].index = y_test.index
            df_forecast[f'{dataset}_low'].index = y_test.index
            df_forecast[f'{dataset}_high'].index = y_test.index
            
            forecast_mean = df_forecast[f'{dataset}_mean']
            forecast_low = df_forecast[f'{dataset}_low']
            forecast_high = df_forecast[f'{dataset}_high']
            
            forecast = pd.concat([y_test, forecast_mean, forecast_low, forecast_high], axis=1)
            forecast.columns = ['actual', 'forecast', 'lower', 'upper']
            
            df_forecast = forecast.assign(
                covered_pts=lambda x:
                    4*x['actual'].between(x['lower'], x['upper'], inclusive='both') +
                    2*(x['actual'].between(2*x['lower']-x['forecast'], x['lower'], inclusive='left') + x['actual'].between(x['upper'], 2*x['upper']-x['forecast'], inclusive='right')) +
                    1*(x['actual'].between(3*x['lower']-2*x['forecast'], 2*x['lower']-x['forecast'], inclusive='left') + x['actual'].between(2*x['upper']-x['forecast'], 3*x['upper']-2*x['forecast'], inclusive='right')),
                covered_width=lambda x: x['upper'] - x['lower'],
            )
            
            df_forecast_metrics = {}
            df_forecast_metrics['total_covered'] = df_forecast['covered_pts'].sum()
            df_forecast_metrics['avg_covered'] = df_forecast['covered_pts'].mean()
            df_forecast_metrics['avg_covered_width'] = df_forecast['covered_width'].mean()
            
            if dataset == 'orders':
                df_forecast_orders_metrics_norm = df_forecast_metrics['avg_covered'] / (1 + np.log(1+df_forecast_metrics['avg_covered_width']))
            else:
                df_forecast_sales_metrics_norm = df_forecast_metrics['avg_covered'] / (1 + np.log(1+df_forecast_metrics['avg_covered_width']))
        
        weight_chan = weights_chan_df.loc[(weights_chan_df['account_id'] == account_id) & (weights_chan_df['sales_channel_id'] == int(sales_channel_id)), 'weights'].values[0]
        df_forecast_metrics_norm[model][account_id][sales_channel_id] = weight_chan * (1/3*df_forecast_sales_metrics_norm + 2/3*df_forecast_orders_metrics_norm)
        
    scores[model] = np.sum([weights_acc_df[acc_id] * np.sum(list(df_forecast_metrics_norm[model][acc_id].values())) for acc_id in df_forecast_metrics_norm[model].keys()])




GradientBoosting_10
sales 47.669202125
orders 0.085734875
sales 56.97368816666667
orders 0.005812
sales 156.602127125
orders 0.147191125
sales 599.1236120833333
orders 0.22852562499999998
sales 20.95133720833333
orders 0.003479
sales 6711.045224583333
orders 0.9150155833333334
sales 179.04718516666665
orders 0.012879
sales 309.8672923333333
orders 0.49215241666666665
sales 0.0
orders 0.0
sales 427.5401616666666
orders 0.09313179166666667
sales 137.05355908333334
orders 0.06659933333333333
sales 121.09057970833334
orders 0.06300608333333334
sales 2.235588
orders 0.003273
sales 109.04678704166668
orders 0.056692791666666666
sales 1079.2183920416667
orders 0.144975375
sales 420.89998045833335
orders 0.9225365
sales 0.0
orders 0.0
sales 0.0
orders 0.0
sales 2962.0593900416666
orders 0.7167977916666667
sales 13.371590958333334
orders 0.0015555416666666667
sales 49.19008420833333
orders 0.10676175
sales 0.88405625
orders 0.0043805833333333336
sales 10.42432725
orders 0.017105833333333334


In [25]:
scores

{'GradientBoosting_10': np.float64(0.04837370077432962),
 'GradientBoosting_20': np.float64(0.08983507436464401),
 'GradientBoosting_30': np.float64(0.09349213409074955),
 'GradientBoosting_40': np.float64(0.16987421954403664),
 'GradientBoosting_50': np.float64(0.13828766067311277),
 'GradientBoosting_60': np.float64(0.11206200765285951),
 'GradientBoosting_70': np.float64(0.08438830487316965),
 'GradientBoosting_80': np.float64(0.10752557335695043),
 'GradientBoosting_90': np.float64(0.17054264615592468),
 'TBATS_10': np.float64(0.15266579879786793),
 'TBATS_20': np.float64(0.2610297349592931),
 'TBATS_30': np.float64(0.29242758372340993),
 'TBATS_40': np.float64(0.2576577095421148),
 'TBATS_50': np.float64(0.3986554530317775),
 'TBATS_60': np.float64(0.4433025500867279),
 'TBATS_70': np.float64(0.4638853034444387),
 'TBATS_80': np.float64(0.4576866804479057),
 'TBATS_90': np.float64(0.41140202184784536)}